# Figures of the manuscript: Figure 6, Figure 7, Figure 8, Appendix Figures D2, D3, D5, E1, E2, E3, E4, E5

Reproduction notebook for *Addressing the Psychological Small-Sample Problem with Multimodal Large Language Models: Evidence from Multiple Moral Emotion Recognition Study*.


## Setup

In [1]:
import os, sys, json, glob
from collections import Counter
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, PathPatch
from matplotlib.path import Path
from PIL import Image

HERE = os.path.abspath(".")
PUB = os.path.abspath(os.path.join(HERE, ".."))          # publications/llm_moral_emotion
FINAL = os.path.join(PUB, "final")
REPRO = os.path.join(FINAL, "reproducibility")
SA = os.path.join(PUB, "statistical_analysis")
OUT = os.path.join(HERE, "output"); os.makedirs(OUT, exist_ok=True)
for p in (FINAL, REPRO, SA):
    assert os.path.isdir(p), p
print("data folder:", os.path.relpath(FINAL, HERE))


data folder: ../final


## 1. Figure 6 — Classification performance and strategy comparisons (Exp 1–15)

(A) Four-class accuracy with 95% bootstrap CI and permutation-test significance; (B) two-dimensional versus four-class accuracy; (C) GEE odds ratios of the 13 model configurations against the vision-only baseline (Appendix E.5).

In [2]:
mpl.rcdefaults(); plt.close("all")
import os, sys, zipfile, shutil, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

BLUE, ORANGE, GREY, GOLD, MLLMC, OBSC = "#3B6FB0", "#E08A2B", "#8C8C8C", "#C9A227", "#C56A2C", "#4D4D4D"

F = 1.5
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12*F, "axes.titlesize": 13*F,
                     "axes.labelsize": 12*F, "xtick.labelsize": 10.5*F, "ytick.labelsize": 10.5*F})

def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."

LBL = {1: "1 Vision", 2: "2 V+A", 3: "3 V+A+T", 4: "4 V+A+T+P", 5: "5 CoT",
       6: "6 KnowInj", 7: "7 FewShot", 8: "8 FineTune", 9: "9 Agentic",
       10: "10 ML V", 11: "11 ML V+A", 12: "12 ML V+A+T", 13: "13 ML V+A+T+P",
       14: "14 Observers, silent", 15: "15 Observers, AV"}

ordA = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

SIG = pd.read_csv(os.path.join(REPRO, "significance_frozen.csv"))

GEE_frozen = pd.read_csv(os.path.join(REPRO, "gee_frozen.csv"))
G = GEE_frozen[GEE_frozen.outcome == "correct"].reset_index(drop=True)

hs = [Line2D([], [], marker="o", color=GREY, ls="", label="Four-class", ms=8),
      Line2D([], [], marker="D", color=MLLMC, ls="", label="Two-dimensional, MLLM", ms=8),
      Line2D([], [], marker="D", color=GOLD, ls="", label="Two-dimensional, Random Forest", ms=8),
      Line2D([], [], marker="D", color=OBSC, ls="", label="Two-dimensional, observers", ms=8)]

fig = plt.figure(figsize=(20, 22))
gs = fig.add_gridspec(2, 2, hspace=0.32, wspace=0.42)
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axL = fig.add_subplot(gs[1, 1])
axL.axis("off")

for a in (axA, axB, axC):
    a.set_box_aspect(0.95)

y = np.arange(len(ordA))[::-1]

for yi, e in zip(y, ordA):
    r = SIG[SIG.exp == e].iloc[0]
    sig = r.p_perm < .05
    c = (GOLD if 10 <= e <= 13 else (OBSC if e >= 14 else MLLMC)) if sig else "#B0B0B0"
    axA.plot([r.acc_lo, r.acc_hi], [yi, yi], color=c, lw=2.5)
    axA.plot(r.acc, yi, "o", color=c, ms=9)
    axA.text(r.acc_hi + 0.02, yi, stars(r.p_perm), va="center", fontsize=11*F, color="#444")

axA.axvline(.25, ls="--", color="#999", lw=1.2)
axA.set_yticks(y)
axA.set_yticklabels([LBL[e] for e in ordA])
axA.set_xlim(0, 1.05)
axA.set_ylim(-0.7, len(ordA))
axA.set_xlabel("Four-class accuracy [95% bootstrap CI]")
axA.set_title("(A) Four-class accuracy against chance", loc="left")

for yi, e in zip(y, ordA):
    r = SIG[SIG.exp == e].iloc[0]
    c = GOLD if 10 <= e <= 13 else (OBSC if e >= 14 else MLLMC)
    axB.plot([r.acc, r.dim_acc], [yi, yi], color="#CCCCCC", lw=2, zorder=1)
    axB.plot(r.acc, yi, "o", color=GREY, ms=7, zorder=2)
    axB.plot(r.dim_acc, yi, "D", color=c, ms=7, zorder=2)

axB.axvline(.25, ls="--", color="#999", lw=1.2)
axB.axvline(.5, ls=":", color="#999", lw=1.2)
axB.set_yticks(y)
axB.set_yticklabels([LBL[e] for e in ordA])
axB.set_xlim(.15, .95)
axB.set_ylim(-0.7, len(ordA))
axB.set_xlabel("Accuracy\n(dashed: four-class chance; dotted: axis chance)")
axB.set_title("(B) Two-dimensional versus four-class accuracy", loc="left")

yc = np.arange(len(G))[::-1]
for yi, (_, r) in zip(yc, G.iterrows()):
    c = MLLMC if r.exp <= 9 else GOLD
    axC.plot([r.ci_lo, r.ci_hi], [yi, yi], color=c, lw=2.5)
    axC.plot(r.OR, yi, "o", color=c, ms=9)
    axC.text(r.ci_hi * 1.08, yi, f"{r.OR:.2f} {stars(r.p)}", va="center", fontsize=10*F)

axC.axvline(1, ls="--", color="#999", lw=1.2)
axC.set_yticks(yc)
axC.set_yticklabels([f"{int(r.exp)} {r['name']}" for _, r in G.iterrows()])
axC.set_xscale("log")
axC.set_xlim(0.4, 20)
axC.set_xticks([0.5, 1, 2, 4, 8, 16])
axC.set_xticklabels(["0.5", "1", "2", "4", "8", "16"])
axC.set_xlabel("Odds ratio vs. vision-only baseline (log; GEE, 95% CI)")
axC.set_title("(C) Within-participant strategy effects, Exp 1–13", loc="left")

for a in (axA, axB, axC):
    a.spines[["top", "right"]].set_visible(False)

axL.legend(handles=hs, frameon=False, loc="center", fontsize=11*F)

pA = axA.get_position()
pC = axC.get_position()
h = pC.height
new_top = pA.y0 - 0.05
axC.set_position([pC.x0, new_top - h, pC.width, h])
pB = axB.get_position()
axL.set_position([pB.x0, new_top - h, pB.width, h])

fig.savefig(os.path.join(OUT, "fig6_results_accuracy.png"), dpi=200, bbox_inches="tight")


## 2. Figure 7 — Criterion validity against self-report

(A) Spearman correlations of the Exp 9 model scores with the two self-report items; (B) convergent correlations across the nine MLLM configurations (Appendix E.3).

In [3]:
mpl.rcdefaults(); plt.close("all")
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

SIG = pd.read_csv(os.path.join(REPRO, "significance_frozen.csv"))

sv = pd.read_csv(os.path.join(REPRO, "selfreport_validity_frozen.csv"))


SIGi = SIG.set_index("exp")

BLUE, ORANGE, GREY, GOLD, MLLMC = "#3B6FB0", "#E08A2B", "#8C8C8C", "#C9A227", "#C56A2C"

zo = sv[(sv.analysis == "zero_order") & (sv.cell == "all")]

def rho(e, ms, sr):
    r = zo[(zo.exp == e) & (zo.model_score == ms) & (zo.self_report == sr)]
    return float(r.rho.iloc[0])

Mx = np.array([[rho(9, "symp", "self_sympathy"), rho(9, "symp", "self_inspired")],
               [rho(9, "insp", "self_sympathy"), rho(9, "insp", "self_inspired")]])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw=dict(width_ratios=[1, 1.15]))
im = a1.imshow(Mx, cmap="RdBu_r", vmin=-0.8, vmax=0.8)
for i_ in range(2):
    for j_ in range(2):
        a1.text(j_, i_, f"{Mx[i_, j_]:.2f}", ha="center", va="center", fontsize=13,
                color="white" if abs(Mx[i_, j_]) > 0.45 else "black",
                fontweight="bold" if i_ == j_ else "normal")
a1.set_xticks([0, 1])
a1.set_xticklabels(["Sympathy", "Inspired"])
a1.set_yticks([0, 1])
a1.set_yticklabels(["Model\nSympathy", "Model\nInspiration"])
a1.set_xlabel("Self-reported experienced emotion")
a1.set_title("(A) Model scores track participants' own reports (Exp 9)", loc="left", fontsize=11)
cb = fig.colorbar(im, ax=a1, fraction=0.046, pad=0.04)
cb.set_label("Spearman ρ")

cfg = [(1, "V"), (2, "V+A"), (3, "V+A+T"), (4, "V+A+T+P"), (5, "CoT"),
       (6, "KnowInj"), (7, "FewShot"), (8, "FineTune"), (9, "Agentic")]
y = np.arange(len(cfg))[::-1]
for yi, (e, l) in zip(y, cfg):
    rs, ri = rho(e, "symp", "self_sympathy"), rho(e, "insp", "self_inspired")
    a2.plot([rs, ri], [yi, yi], color="#CCCCCC", lw=1.5, zorder=1)
    a2.plot(rs, yi, "o", color=BLUE, ms=8, zorder=2)
    a2.plot(ri, yi, "D", color=MLLMC, ms=7, zorder=2)
a2.axvline(0, color="#999", lw=1)
a2.set_yticks(y)
a2.set_yticklabels([l for _, l in cfg])
a2.set_xlim(-0.1, 0.9)
a2.set_xlabel("Convergent ρ (model vs self-report)")
a2.set_title("(B) Criterion validity strengthens with audio and text", loc="left", fontsize=11)
a2.legend(handles=[Line2D([], [], marker="o", color=BLUE, ls="", label="Sympathy", ms=8),
                   Line2D([], [], marker="D", color=MLLMC, ls="", label="Inspiration", ms=7)],
          frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
for a in (a1, a2):
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "fig_criterion_validity.png"), dpi=250, bbox_inches="tight")
plt.close(fig)


## 3. Figure 8 — Cue–class associations in the Exp 4 explanations

(A) Sankey of cue → predicted class → induced group; (B) feature–class probability lift; (C) cue importance (Appendix E.6).

In [4]:
mpl.rcdefaults(); plt.close("all")
import sys
from collections import Counter
import numpy as np, pandas as pd, matplotlib
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, PathPatch
from matplotlib.path import Path

FC = pd.read_csv(os.path.join(FINAL, "exp4_feature_coding.csv"))
PH = pd.read_csv(os.path.join(FINAL, "exp4_physio_direction_coding.csv"))
D_ = FC.drop(columns=[c for c in FC.columns if c.startswith("phys_") or c == "cues_present"]).merge(PH, on="pid")
for c in ("true", "pred"): D_[c] = D_[c].replace({"ADMR": "INSP"})

GRP = ["ELVT", "SYMP", "INSP", "CTRL"]
GN = {"ELVT": "Elevation", "SYMP": "Sympathy", "INSP": "Inspiration", "CTRL": "Control"}
CC = {"text": "#3B6FB0", "nonverbal": "#E08A2B", "physio": "#C9A227"}
GC = {"ELVT": "#7B5EA7", "SYMP": "#2A9D8F", "INSP": "#C0504D", "CTRL": "#6B6B6B"}
NICE = {"text_sad": "Sad/distress language", "text_anger": "Anger language", "text_touched": '"Touched" language',
        "text_pride": "Pride/excitement language", "text_admire": "Praise language", "face_tears": "Tears",
        "face_frown": "Frown", "face_smile": "Smile", "face_neutral": "Neutral expression",
        "vocal_subdued": "Subdued voice", "vocal_excited": "Excited voice", "body_still": "Still posture",
        "body_animated": "Animated gestures",
        "cheek_up": "Cheek temperature ↑", "cheek_down": "Cheek temperature ↓",
        "forehead_up": "Forehead temperature ↑", "forehead_down": "Forehead temperature ↓",
        "nose_up": "Nose/perioral temperature ↑", "nose_down": "Nose/perioral temperature ↓",
        "face_temp_up_other": "Other facial temperature ↑", "face_temp_down_other": "Other facial temperature ↓",
        "hr_up": "Heart rate ↑", "hr_down": "Heart rate ↓", "hrv_up": "HRV ↑", "hrv_down": "HRV ↓",
        "phys_nodirection": "Physiology, direction unstated"}
PHY = list(PH.columns[1:])
CH = lambda c: "text" if c.startswith("text_") else ("physio" if c in PHY else "nonverbal")
CUES = sorted([c for c in NICE if c in D_.columns and D_[c].sum() > 0],
              key=lambda c: (["text", "nonverbal", "physio"].index(CH(c)), -D_[c].sum()))

labels = [f"{NICE[c]} ({int(D_[c].sum())})" for c in CUES] + \
         [f"Predicted {GN[g]} ({(D_.pred == g).sum()})" for g in GRP] + \
         [f"Induced {GN[g]} ({(D_.true == g).sum()})" for g in GRP]
ncue = len(CUES); NN = len(labels); xs = [0] * ncue + [1] * 4 + [2] * 4
src, tgt, val = [], [], []
for i, c in enumerate(CUES):
    for j, g in enumerate(GRP):
        v = int(D_.loc[D_.pred == g, c].sum())
        if v: src.append(i); tgt.append(ncue + j); val.append(v)
for j, g in enumerate(GRP):
    for k, h in enumerate(GRP):
        v = int(((D_.pred == g) & (D_.true == h)).sum())
        if v: src.append(ncue + j); tgt.append(ncue + 4 + k); val.append(v)
ncol = [CC[CH(c)] for c in CUES] + [GC[g] for g in GRP] * 2
hexcol = [CC[CH(CUES[s])] if s < ncue else (GC[GRP[s - ncue]] if s - ncue == t - ncue - 4 else "#C8C8C8")
          for s, t in zip(src, tgt)]
tot_in, tot_out = Counter(), Counter()
for s, t, v in zip(src, tgt, val): tot_out[s] += v; tot_in[t] += v
size = {i: max(tot_in[i], tot_out[i]) for i in range(NN)}; pos = {}
for col in (1, 2):
    ids = [i for i in range(NN) if xs[i] == col]; gap = .03; sc = (1 - gap * 3) / sum(size[i] for i in ids); y = 1
    for i in ids: h = size[i] * sc; pos[i] = (y - h, y); y -= h + gap
gap, mh = .008, .02; tot = sum(size[i] for i in range(ncue))
hs_ = {i: max(size[i] / tot, mh) for i in range(ncue)}; f = (1 - gap * (ncue - 1)) / sum(hs_.values()); y = 1
for i in range(ncue): h = hs_[i] * f; pos[i] = (y - h, y); y -= h + gap

def band(ax, x0, y0a, y0b, x1, y1a, y1b, color):
    xm = (x0 + x1) / 2
    verts = [(x0, y0a), (xm, y0a), (xm, y1a), (x1, y1a), (x1, y1b), (xm, y1b), (xm, y0b), (x0, y0b), (x0, y0a)]
    codes = [Path.MOVETO] + [Path.CURVE4] * 3 + [Path.LINETO] + [Path.CURVE4] * 3 + [Path.CLOSEPOLY]
    ax.add_patch(PathPatch(Path(verts, codes), fc=color, ec="none", alpha=.45))

plt.rcdefaults()
mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 15, "axes.labelsize": 15,
                     "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 14})
FS, TS = 14.5, 18

def sankey_ax(ax):
    X = [0.0, 0.45, 0.9]; W_ = 0.018
    offo = {i: pos[i][1] for i in range(NN)}; offi = dict(offo)
    for k in sorted(range(len(src)), key=lambda k: (xs[src[k]], -pos[src[k]][1], -pos[tgt[k]][1])):
        s, t, v = src[k], tgt[k], val[k]
        hs = v / tot_out[s] * (pos[s][1] - pos[s][0]); ht = v / tot_in[t] * (pos[t][1] - pos[t][0])
        a0 = offo[s]; offo[s] -= hs; b0 = offi[t]; offi[t] -= ht
        band(ax, X[xs[s]] + W_, a0, a0 - hs, X[xs[t]], b0, b0 - ht, hexcol[k])
    for i in range(NN):
        y0, y1 = pos[i]; ax.add_patch(Rectangle((X[xs[i]], y0), W_, y1 - y0, color=ncol[i]))
        left = xs[i] == 0
        ax.text(X[xs[i]] - 0.006 if left else X[xs[i]] + W_ + 0.006, (y0 + y1) / 2, labels[i],
                ha="right" if left else "left", va="center", fontsize=FS)
    ax.set_xlim(-0.02, 1.0); ax.set_ylim(-0.005, 1.06); ax.axis("off")
    for x, t in zip(X, ["Cue code (explanations citing it)", "Predicted class", "Induced group"]):
        ax.text(x + W_ / 2, 1.025, t, ha="center" if x else "right", va="bottom", fontsize=FS + 1, fontweight="bold")

base = {g: (D_.pred == g).mean() for g in GRP}
LIFT = pd.DataFrame({g: [D_.loc[D_[c] == 1, "pred"].eq(g).mean() - base[g] for c in CUES] for g in GRP}, index=CUES)
IMP = D_[CUES].mean() * LIFT.abs().sum(1)

fig = plt.figure(figsize=(14, 22))
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 0.95], hspace=0.21, wspace=0.95, left=0.30, right=0.98, top=0.97, bottom=0.04)
axA = fig.add_subplot(gs[0, :]); sankey_ax(axA)
axA.legend(handles=[Rectangle((0, 0), 1, 1, color=CC[c], alpha=.7, label=f"{c} cue") for c in CC],
           loc="upper left", bbox_to_anchor=(-0.28, -0.012), ncol=3, frameon=False, fontsize=14)
axB = fig.add_subplot(gs[1, 0]); L = LIFT.loc[CUES, GRP].values
cmapb = mpl.colors.LinearSegmentedColormap.from_list("b", ["#F4F7FB", "#3B6FB0", "#17365D"])
im = axB.imshow(np.abs(L), cmap=cmapb, vmin=0, vmax=0.3, aspect="auto")
for i in range(L.shape[0]):
    for j in range(4):
        axB.text(j, i, f"{L[i, j]:+.2f}".replace("+0.", ".").replace("-0.", "−."), ha="center", va="center",
                 fontsize=12.5, color="white" if abs(L[i, j]) > 0.18 else "black")
axB.set_xticks(range(4)); axB.set_xticklabels(GRP); axB.set_yticks(range(len(CUES)))
axB.set_yticklabels([NICE[c] for c in CUES])
for tl, c in zip(axB.get_yticklabels(), CUES): tl.set_color(CC[CH(c)])
axB.set_xlabel("Predicted class"); axB.xaxis.set_label_position("top"); axB.xaxis.tick_top()
cb = fig.colorbar(im, ax=axB, orientation="horizontal", fraction=0.035, pad=0.07, aspect=25)
cb.set_label("|Lift| (sign shown in each cell)")
axC = fig.add_subplot(gs[1, 1]); I_ = IMP.sort_values()
axC.barh([NICE[c] for c in I_.index], I_.values, color=[CC[CH(c)] for c in I_.index])
axC.set_xlabel("Feature importance\n(prevalence × Σ|class lift|)"); axC.spines[["top", "right"]].set_visible(False)
axC.set_ylim(-0.6, len(I_) - 0.4); axC.tick_params(axis="y", pad=2)
axC.legend(handles=[Rectangle((0, 0), 1, 1, color=CC[c], label=c) for c in CC], frameon=False, loc="lower right")

fig.canvas.draw(); r = fig.canvas.get_renderer(); inv = fig.transFigure.inverted()
bbA = axA.get_tightbbox(r).transformed(inv); bbB = axB.get_tightbbox(r).transformed(inv); bbC = axC.get_tightbbox(r).transformed(inv)
xL = min(bbA.x0, bbB.x0)
fig.text(xL, axA.get_position().y1 + 0.012, "(A) Cues cited in the Exp 4 explanations, predicted class, and induced group",
         fontsize=TS, ha="left", va="bottom")
yBC = max(bbB.y1, axC.get_position().y1) + 0.034
fig.text(xL, yBC, "(B) Feature–class probability lift\n      P(class | cue) − base rate, n = 100", fontsize=TS, ha="left", va="top")
fig.text(bbC.x0, yBC, "(C) Which cues best predict class", fontsize=TS, ha="left", va="top")
fig.savefig(os.path.join(OUT, "fig_new_interp.png"), dpi=200, bbox_inches="tight"); plt.close(fig)


## 4. Appendix Figure D2 — Speech semantics versus physiology over the V+A base

(A) Four-class accuracy of Exp 2, 3 and 16; (B) participants improved / worsened relative to Exp 2 (GEE contrast for Exp 3; paired bootstrap descriptives for Exp 16, Appendix E.5.4).

In [5]:
mpl.rcdefaults(); plt.close("all")
SIG = pd.read_csv(os.path.join(REPRO, "significance_frozen.csv")).set_index("exp")
CON = pd.read_csv(os.path.join(REPRO, "gee_contrasts_frozen.csv"))
LABF = pd.read_csv(os.path.join(REPRO, "labels_all_frozen.csv"))
LAB = {e: g.reset_index(drop=True) for e, g in LABF.groupby("exp")}

# Within-participant descriptives for the V+A (Exp 2) -> V+A+P (Exp 16) comparison, which is not a GEE contrast:
# paired participants, improved / worsened counts and the bootstrap CI of the accuracy difference (same procedure
# and seed as common.paired_descriptives in ../statistical_analysis).
sys.path.insert(0, SA)
import common as C
d216 = C.paired_descriptives(LAB, 2, 16, np.random.default_rng(C.SEED))
r23 = CON[(CON.exp_a == 2) & (CON.exp_b == 3)].iloc[0]

BLUE, ORANGE, GREY, GOLD, MLLMC = "#3B6FB0", "#E08A2B", "#8C8C8C", "#C9A227", "#C56A2C"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})

def pf(p):
    s = "< .001" if p < .001 else f"= {p:.3f}" if p < .01 else f"= {p:.2f}"
    return s.replace("0.", ".")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))

# (A) four-class accuracy of the three configurations
E = [2, 3, 16]
LB = ["V+A\n(Exp 2)", "V+A+T\n(+ speech semantics, Exp 3)", "V+A+P\n(+ physiology, Exp 16)"]
COL = [GREY, MLLMC, BLUE]
for i, e in enumerate(E):
    a1.bar(i, SIG.loc[e, "acc"], 0.6, color=COL[i])
    a1.text(i, SIG.loc[e, "acc"] + 0.012, f"{SIG.loc[e, 'acc']:.2f}".replace("0.", "."), ha="center", fontsize=12)
a1.axhline(0.25, ls=":", color="#999", lw=2)
a1.text(2.9, 0.26, "chance", ha="right", color="#999", fontsize=10)
a1.plot([0, 0, 1, 1], [0.70, 0.715, 0.715, 0.70], color="#333", lw=1)
a1.text(0.5, 0.72, f"OR = {r23.OR:.2f}; Holm p {pf(r23.p_holm)}", ha="center", fontsize=11)
a1.set_xticks(range(3)); a1.set_xticklabels(LB, fontsize=9)
a1.set_ylim(0, 0.78); a1.set_ylabel("Four-class accuracy")
a1.set_title("Adding speech semantics, not physiology, raises accuracy over the V+A base", loc="left", fontsize=11)

# (B) participants whose classification changed relative to V+A
x = np.arange(2); w = 0.35
imp = [r23.improved, d216["improved"]]; wor = [r23.worsened, d216["worsened"]]
a2.bar(x - w / 2, imp, w, color=BLUE, label="improved")
a2.bar(x + w / 2, wor, w, color="#AAAAAA", label="worsened")
for xi, v in zip(x - w / 2, imp): a2.text(xi, v + 0.4, str(int(v)), ha="center", fontsize=12)
for xi, v in zip(x + w / 2, wor): a2.text(xi, v + 0.4, str(int(v)), ha="center", fontsize=12)
a2.text(0.35, 26.2, f"OR = {r23.OR:.2f}, p {pf(r23.p_gee)}", ha="center", fontsize=11)
a2.text(1, 26.2, f"Δ = {d216['diff']:+.2f} [{d216['diff_lo']:+.2f}, {d216['diff_hi']:+.2f}]".replace("0.", "."), ha="center", fontsize=11)
a2.set_xticks(x); a2.set_xticklabels(["+ speech semantics\n(Exp 3 vs. Exp 2)", "+ physiology\n(Exp 16 vs. Exp 2)"], fontsize=9)
a2.set_ylim(0, 30); a2.set_ylabel("Participants whose classification changed")
a2.set_title("Within-participant shifts relative to the V+A base", loc="left", fontsize=11)
a2.legend(frameon=False, loc="upper left")
for a in (a1, a2):
    a.spines[["top", "right"]].set_visible(False)
a1.text(-0.12, 1.02, "A", transform=a1.transAxes, fontsize=14, fontweight="bold")
a2.text(-0.12, 1.02, "B", transform=a2.transAxes, fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "fig_appendixD_exp16_vs_text.png"), dpi=250, bbox_inches="tight")
plt.close(fig)


## 5. Appendix Figure D3 — Exp 16 compared with Exp 2 and Exp 3

(A) Four-class and two-dimensional accuracy; (B) partial η² of the matching manipulation; (C) criterion validity; (D) per-class recall.

In [6]:
mpl.rcdefaults(); plt.close("all")
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

SIG = pd.read_csv(os.path.join(REPRO, "significance_frozen.csv"))
AN = pd.read_csv(os.path.join(REPRO, "anova_frozen.csv"))
SV = pd.read_csv(os.path.join(REPRO, "selfreport_validity_frozen.csv"))
RC = pd.read_csv(os.path.join(REPRO, "perclass_recall_frozen.csv"))

BLUE, ORANGE, GREY, GOLD, MLLMC = "#3B6FB0", "#E08A2B", "#8C8C8C", "#C9A227", "#C56A2C"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})
CL = ["ELVT", "SYMP", "INSP", "CTRL"]
S = SIG.set_index("exp")

def stars(p): return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."

E = [2, 3, 16]
LB = {2: "V+A\n(Exp 2)", 3: "V+A+T\n(Exp 3)", 16: "V+A+P\n(Exp 16)"}
COL = {2: GREY, 3: MLLMC, 16: BLUE}

fig, ax = plt.subplots(2, 2, figsize=(12, 9))
x = np.arange(3)

a = ax[0, 0]
for i, e in enumerate(E):
    r = S.loc[e]
    a.bar(i - 0.18, r.acc, 0.34, color=COL[e])
    a.errorbar(i - 0.18, r.acc, yerr=[[r.acc - r.acc_lo], [r.acc_hi - r.acc]], color="k", capsize=3)
    a.bar(i + 0.18, r.dim_acc, 0.34, color=COL[e], alpha=0.45, hatch="//", edgecolor="white")
    a.text(i - 0.18, r.acc_hi + 0.02, f"{r.acc:.2f}".lstrip("0"), ha="center", fontsize=9)
    a.text(i + 0.18, r.dim_acc + 0.02, f"{r.dim_acc:.2f}".lstrip("0"), ha="center", fontsize=9)
a.axhline(.25, ls="--", color="#999", lw=1)
a.axhline(.5, ls=":", color="#999", lw=1)
a.set_xticks(x)
a.set_xticklabels([LB[e] for e in E])
a.set_ylim(0, 1)
a.set_ylabel("Accuracy")
a.set_title("(A) Four-class (solid, 95% CI) and two-dimensional (hatched) accuracy", loc="left", fontsize=11)

a = ax[0, 1]
w = 0.38
for i, e in enumerate(E):
    s = AN[(AN.exp == e) & (AN.score == "symp") & (AN.effect == "Sympathy manip")].eta_p2.iloc[0]
    n = AN[(AN.exp == e) & (AN.score == "insp") & (AN.effect == "Inspiration manip")].eta_p2.iloc[0]
    a.bar(i - w / 2, s, w, color=BLUE)
    a.bar(i + w / 2, n, w, color=ORANGE)
    a.text(i - w / 2, s + 0.01, f"{s:.2f}".lstrip("0"), ha="center", fontsize=9)
    a.text(i + w / 2, n + 0.01, f"{n:.2f}".lstrip("0"), ha="center", fontsize=9)
a.set_xticks(x)
a.set_xticklabels([LB[e] for e in E])
a.set_ylim(0, 0.55)
a.set_ylabel("Partial η² of the matching manipulation")
a.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=BLUE, label="Sympathy score ← sympathy manipulation"),
                  plt.Rectangle((0, 0), 1, 1, color=ORANGE, label="Inspiration score ← inspiration manipulation")],
         frameon=False, fontsize=9, loc="upper left")
a.set_title("(B) Design alignment of the continuous scores", loc="left", fontsize=11)

a = ax[1, 0]
z = SV[(SV.analysis == "zero_order") & (SV.cell == "all")]
for i, e in enumerate(E):
    for k, (ms, sr, c) in enumerate([("symp", "self_sympathy", BLUE), ("insp", "self_inspired", ORANGE)]):
        r = z[(z.exp == e) & (z.model_score == ms) & (z.self_report == sr)].iloc[0]
        xx = i + (k - 0.5) * 0.3
        a.errorbar(xx, r.rho, yerr=[[r.rho - r.ci_lo], [r.ci_hi - r.rho]],
                   fmt="o" if k == 0 else "D", color=c, capsize=3, ms=7)
a.axhline(0, color="#999", lw=1)
a.set_xticks(x)
a.set_xticklabels([LB[e] for e in E])
a.set_ylim(-0.1, 0.8)
a.set_ylabel("Spearman ρ with matching self-report [95% CI]")
a.legend(handles=[Line2D([], [], marker="o", color=BLUE, ls="", label="Sympathy"),
                  Line2D([], [], marker="D", color=ORANGE, ls="", label="Inspiration")],
         frameon=False, fontsize=9, loc="upper left")
a.set_title("(C) Criterion validity against self-report", loc="left", fontsize=11)

a = ax[1, 1]
R = RC.set_index(RC.columns[0]).rename(columns={"INSP": "ADMR"})
w = 0.25
for i, e in enumerate(E):
    a.bar(np.arange(4) + (i - 1) * w, R.loc[e, ["ELVT", "SYMP", "ADMR", "CTRL"]].values,
          w, color=COL[e], label=LB[e].replace("\n", " "))
a.set_xticks(range(4))
a.set_xticklabels(CL)
a.set_ylim(0, 1)
a.set_ylabel("Recall")
a.legend(frameon=False, fontsize=9, loc="upper left")
a.set_title("(D) Per-class recall", loc="left", fontsize=11)

for a in ax.flat:
    a.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig(os.path.join(OUT, "fig_appendixD_exp16.png"), dpi=220)
plt.close(fig)


## 6. Appendix Figure D5 — Exp 17 results for the six models and their joint average

Statistics are expanded from the committed confusion matrices (n = 100 per model).

In [7]:
mpl.rcdefaults(); plt.close("all")
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

BLUE, ORANGE, GREY, GOLD, MLLMC = "#3B6FB0", "#E08A2B", "#8C8C8C", "#C9A227", "#C56A2C"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})

import os


# Using the artifact reference from the trace
J = json.load(open(os.path.join(FINAL, "exp17_18_confusion_matrices.json")))
cms = {k: np.array(v, float) for k, v in J["confusion_matrices_exp18"].items()}

SH = {
    "Claude Haiku 4.5": "Claude",
    "Deepseek 4.1 Flash": "Deepseek",
    "GPT6 Luna": "GPT6",
    "Gemini 2.5 Flash": "Gemini 2.5",
    "Gemini 3.8 Flash": "Gemini 3.8",
    "Qwen 3.8 Flash": "Qwen",
    "Joint average": "Joint avg"
}

names = list(cms)
rows = []
for k in names:
    m = cms[k].astype(int)
    t = []; p = []
    for i in range(4):
        for j in range(4):
            t += [i] * m[i, j]; p += [j] * m[i, j]
    t = np.array(t); p = np.array(p)
    hs = lambda v: np.isin(v, [0, 1])
    hi = lambda v: np.isin(v, [0, 2])
    rows.append(dict(
        m=SH[k],
        acc=(t == p).mean(),
        sax=(hs(t) == hs(p)).mean(),
        iax=(hi(t) == hi(p)).mean(),
        sf=f1_score(hs(t), hs(p)),
        if_=f1_score(hi(t), hi(p)),
        sh=hs(p).mean(),
        ih=hi(p).mean(),
        dsh=hs(t).mean(),
        dih=hi(t).mean()
    ))
T = pd.DataFrame(rows)

fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
y = np.arange(len(T))[::-1]
a = ax[0]
a.barh(y, T.acc, color=[MLLMC] * 6 + [GREY])
a.axvline(.25, ls="--", color="#999")
for yi, v in zip(y, T.acc):
    a.text(v + .005, yi, f"{v:.2f}".lstrip("0"), va="center", fontsize=9)
a.set_yticks(y); a.set_yticklabels(T.m)
a.set_xlim(0, .5)
a.set_xlabel("Four-class accuracy (dashed = chance)")
a.set_title("(A) Four-class accuracy by model", loc="left")

a = ax[1]; w = .38
a.barh(y + w / 2, T.sf, w, color=BLUE)
a.barh(y - w / 2, T.if_, w, color=ORANGE)
a.set_yticks(y); a.set_yticklabels([])
a.set_xlim(0, .6)
a.set_xlabel("High-class F1")
a.set_title("(B) High-class F1 by appraisal axis", loc="left")

a = ax[2]
a.barh(y + w / 2, T.sh, w, color=BLUE)
a.barh(y - w / 2, T.ih, w, color=ORANGE)
a.axvline(T.dsh.iloc[0], color=BLUE, ls=":")
a.axvline(T.dih.iloc[0], color=ORANGE, ls=":")
a.set_yticks(y); a.set_yticklabels([])
a.set_xlim(0, .6)
a.set_xlabel("Share predicted high (dotted = design share)")
a.set_title("(C) Models rarely predict high Inspiration", loc="left")

for a in ax:
    a.spines[["top", "right"]].set_visible(False)

fig.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color=BLUE, label="Sympathy axis"),
        plt.Rectangle((0, 0), 1, 1, color=ORANGE, label="Inspiration axis")
    ],
    loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.62, -0.01)
)
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(os.path.join(OUT, "fig_appendixD_exp17.png"), dpi=220)
plt.close(fig)


## 7. Appendix Figure E1 — Class-specific recall by configuration (Exp 1–15)

In [8]:
mpl.rcdefaults(); plt.close("all")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors

# Class-specific recall of every configuration, from the frozen per-class recall table (Exp 1-13 and 16 are
# recomputed from labels_all_frozen.csv by ../statistical_analysis/01_E1_...; the two observer passes come from the
# cleaned rating panel, so this figure reads the frozen table directly).
RC = pd.read_csv(os.path.join(REPRO, "perclass_recall_frozen.csv"))
RECALL = RC.set_index(RC.columns[0])[["ELVT", "SYMP", "INSP", "CTRL"]].rename(columns={"INSP": "ADMR"})
CLS = ["ELVT", "SYMP", "ADMR", "CTRL"]
OLD = CLS


order = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
LBL = {
    1: "Baseline (Exp 1)", 2: "V+A (Exp 2)", 3: "V+A+T (Exp 3)",
    4: "V+A+T+P (Exp 4)", 5: "CoT (Exp 5)", 6: "KnowInj (Exp 6)",
    7: "FewShot (Exp 7)", 8: "FineTune (Exp 8)", 9: "Agentic (Exp 9)",
    10: "ML V (Exp 10)", 11: "ML V+A (Exp 11)",
    12: "ML V+A+T (Exp 12)", 13: "ML V+A+T+P (Exp 13)",
    14: "Observers, silent (Exp 14)", 15: "Observers, audiovisual (Exp 15)"
}

cmapB = matplotlib.colors.LinearSegmentedColormap.from_list("blues", ["#F4F7FB", "#3B6FB0", "#17365D"])

# Use INSP as the relabeled version of ADMR
display_CLS = ["ELVT", "SYMP", "INSP", "CTRL"]

M_ = RECALL.loc[order, OLD].values.astype(float)

fig, ax = plt.subplots(figsize=(7.2, 8.2))
im = ax.imshow(M_, cmap=cmapB, vmin=0, vmax=1, aspect="auto")
for i in range(len(order)):
    w = int(np.argmin(M_[i]))
    for j in range(4):
        ax.text(j, i, f"{M_[i, j]:.2f}".replace("0.", "."), ha="center", va="center",
                fontsize=9, color="white" if M_[i, j] > 0.55 else "black")
    ax.add_patch(plt.Rectangle((w - 0.5, i - 0.5), 1, 1, fill=False, lw=2, edgecolor="#1a1a1a"))
ax.set_xticks(range(4))
ax.set_xticklabels(display_CLS)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([LBL[e] for e in order], fontsize=9)
ax.set_xlabel("True class")
for yy in (9.5, 13.5):
    ax.axhline(yy, color="#1a1a1a", lw=1.2)
ax.set_title("Per-class recall by configuration\n(boxed = weakest class in each row)", loc="left", fontsize=11)
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
cb.set_label("Recall")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "fig_appendix_perclass_recall.png"), dpi=250)
plt.close(fig)


## 8. Appendix Figure E2 — Unimodal Random Forest classifiers (five-fold GroupKFold)

In [9]:
mpl.rcdefaults(); plt.close("all")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
import os, sys, glob

def apply_figure_style(*, frame="open", font=None, sizes=(8, 7, 6), grid=False):
    if frame not in ("open", "boxed", "none"):
        raise ValueError(f"frame must be 'open'|'boxed'|'none', got {frame!r}")
    try:
        import matplotlib.font_manager as fm
        fdir = os.path.join(os.environ.get("CONDA_PREFIX") or sys.prefix, "fonts")
        if os.path.isdir(fdir):
            known = {f.fname for f in fm.fontManager.ttflist}
            for f in glob.glob(os.path.join(fdir, "*.ttf")):
                if f not in known:
                    fm.fontManager.addfont(f)
    except Exception:
        pass
    base, secondary, tick = sizes
    boxed = (frame == "boxed")
    rc = {
        "font.family": "sans-serif",
        "font.size": base,
        "axes.labelsize": base,
        "axes.titlesize": base,
        "legend.fontsize": secondary,
        "xtick.labelsize": tick,
        "ytick.labelsize": tick,
        "axes.linewidth": 0.6,
        "xtick.direction": "out", "ytick.direction": "out",
        "xtick.major.size": 3, "ytick.major.size": 3,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "axes.spines.top": boxed, "axes.spines.right": boxed,
        "axes.spines.left": frame != "none", "axes.spines.bottom": frame != "none",
        "axes.grid": bool(grid),
        "legend.frameon": False,
        "figure.dpi": 200,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    }
    mpl.rcParams.update(rc)

def panel_letter(ax, letter, case="upper", dx=0, dy=0):
    l = letter.upper() if case == "upper" else letter.lower()
    ax.text(dx, 1.0 + dy, l, transform=ax.transAxes, fontsize=12, fontweight="bold",
            va="bottom", ha="left")

apply_figure_style()

CLS = ["ELVT", "SYMP", "INSP", "CTRL"]
GOLD = "#C9A227"

ma = pd.read_csv(os.path.join(REPRO, "ml_modality_axis_5fold.csv"))
mb = pd.read_csv(os.path.join(REPRO, "ml_modality_recall_5fold.csv")).set_index("modality").rename(columns={"INSP": "ADMR"})

NICEM = {"thermal": "Thermal", "Facial": "Facial", "r-PPG": "rPPG", "Speech Text": "Speech text",
         "pose": "Pose", "Accoustics": "Acoustics", "Eye_Movement": "Eye movement"}

ma = ma.sort_values("acc", ascending=True)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw=dict(width_ratios=[1, 1.1]))

a1.barh([NICEM[m] for m in ma.modality], ma.acc, color=GOLD)
for i, (v, n) in enumerate(zip(ma.acc, ma.N)):
    a1.text(v + 0.008, i, f"{v:.2f}".replace("0.", ".") + f"  (n = {n})", va="center", fontsize=8)
a1.axvline(0.25, ls=":", color="#999")
a1.set_xlim(0, 0.6)
a1.set_xlabel("Four-class accuracy (five-fold GroupKFold Random Forest)")
a1.set_title("Thermal, facial and rPPG features carry the most signal; dotted = chance", loc="left")

cmap = mpl.colors.LinearSegmentedColormap.from_list("blues", ["#F4F7FB", "#3B6FB0", "#17365D"])
mods = list(ma.modality[::-1])
M_ = mb.loc[mods, ["ELVT", "SYMP", "ADMR", "CTRL"]].values

im = a2.imshow(M_, cmap=cmap, vmin=0, vmax=0.7, aspect="auto")
for i in range(len(mods)):
    for j in range(4):
        a2.text(j, i, f"{M_[i, j]:.2f}".replace("0.", "."), ha="center", va="center", fontsize=8,
                color="white" if M_[i, j] > 0.4 else "black")
a2.set_xticks(range(4))
a2.set_xticklabels(["ELVT", "SYMP", "INSP", "CTRL"])
a2.set_yticks(range(len(mods)))
a2.set_yticklabels([NICEM[m] for m in mods])
a2.set_xlabel("True class")
a2.set_title("Each modality recovers different classes; Elevation is weak everywhere", loc="left")
cb = fig.colorbar(im, ax=a2, fraction=0.04, pad=0.03)
cb.set_label("Recall")

panel_letter(a1, "A", case="upper", dx=-0.25)
panel_letter(a2, "B", case="upper", dx=-0.22)

fig.tight_layout()
fig.savefig(os.path.join(OUT, "fig_appendix_ml_modality.png"), dpi=250)


## 9. Appendix Figure E3 — Design alignment of the continuous model scores

Partial η² from the Type II two-way ANOVAs (Table A2).

In [10]:
mpl.rcdefaults(); plt.close("all")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import os, sys, glob

def apply_figure_style(*, frame="open", font=None, sizes=(8, 7, 6), grid=False):
    if frame not in ("open", "boxed", "none"):
        raise ValueError(f"frame must be 'open'|'boxed'|'none', got {frame!r}")
    try:
        import matplotlib.font_manager as fm
        fdir = os.path.join(os.environ.get("CONDA_PREFIX") or sys.prefix, "fonts")
        if os.path.isdir(fdir):
            known = {f.fname for f in fm.fontManager.ttflist}
            for f in glob.glob(os.path.join(fdir, "*.ttf")):
                if f not in known:
                    fm.fontManager.addfont(f)
    except Exception:
        pass
    base, secondary, tick = sizes
    boxed = (frame == "boxed")
    rc = {
        "font.family": "sans-serif",
        "font.size": base,
        "axes.labelsize": base,
        "axes.titlesize": base,
        "legend.fontsize": secondary,
        "xtick.labelsize": tick,
        "ytick.labelsize": tick,
        "axes.linewidth": 0.6,
        "xtick.direction": "out", "ytick.direction": "out",
        "xtick.major.size": 3, "ytick.major.size": 3,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "axes.spines.top": boxed, "axes.spines.right": boxed,
        "axes.spines.left": frame != "none", "axes.spines.bottom": frame != "none",
        "axes.grid": bool(grid),
        "legend.frameon": False,
        "figure.dpi": 200,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    }
    mpl.rcParams.update(rc)

def panel_letter(ax, letter, case="upper", dx=0, dy=0):
    l = letter.upper() if case == "upper" else letter.lower()
    ax.text(dx, 1.02 + dy, l, transform=ax.transAxes, fontsize=11, fontweight="bold", va="bottom", ha="left")

BLUE = "#3B6FB0"
ORANGE = "#E08A2B"
GREY = "#8C8C8C"
GOLD = "#C9A227"

AN = pd.read_csv(os.path.join(REPRO, "anova_frozen.csv"))
AN["effect"] = AN["effect"].replace({"Admiration manip": "Inspiration manip"})

plt.close("all")
apply_figure_style()
exps = [1, 2, 3, 4, 5, 6, 7, 8, 9]
lab = [f"Exp {e}" for e in exps]
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, score, ttl in [(axs[0], "symp", "Sympathy score responds to the sympathy manipulation (all but Exp 1)"),
                       (axs[1], "insp", "Inspiration score responds to the inspiration manipulation (all but Exp 1–2)")]:
    x = np.arange(len(exps)); w = 0.27
    for k, (eff, cc, l) in enumerate([("Sympathy manip", BLUE, "Sympathy manipulation"),
                                       ("Inspiration manip", ORANGE, "Inspiration manipulation"),
                                       ("Interaction", GREY, "Interaction")]):
        v = [AN[(AN.exp == e) & (AN.score == score) & (AN.effect == eff)].eta_p2.iloc[0] for e in exps]
        ax.bar(x + (k - 1) * w, v, w, color=cc, label=l)
    ax.set_xticks(x)
    ax.set_xticklabels(lab, rotation=45, ha="right")
    ax.set_title(ttl, loc="left")
    ax.set_ylim(0, 0.8)
axs[0].set_ylabel("Partial η² (Type II two-way ANOVA)")
axs[0].legend(frameon=False, loc="upper left")
panel_letter(axs[0], "A", case="upper", dx=-0.12)
panel_letter(axs[1], "B", case="upper", dx=-0.05)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "figE3_design_alignment.png"), dpi=250)
plt.close(fig)


## 10. Appendix Figure E4 — Per-dimension accuracy on the sympathy and inspiration axes (Exp 1–13)

In [11]:
mpl.rcdefaults(); plt.close("all")
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

SIG_PATH = os.path.join(REPRO, "significance_frozen.csv")
SIG = pd.read_csv(SIG_PATH)

MLLMC = "#C56A2C"
BLUE = "#3B6FB0"

order = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
LBL = {
    1: "1 Base", 2: "2 V+A", 3: "3 V+A+T", 4: "4 V+A+T+P",
    5: "5 CoT", 6: "6 KnowInj", 7: "7 FewShot", 8: "8 FineTune",
    9: "9 Agentic", 10: "10 ML-V", 11: "11 ML-V+A",
    12: "12 ML-V+A+T", 13: "13 ML-V+A+T+P"
}

SIGi = SIG.set_index("exp")

fig, ax = plt.subplots(figsize=(5.8, 6.2))
y = np.arange(len(order))[::-1]
for yi, e in zip(y, order):
    ax.plot(SIGi.loc[e, "symp_axis"], yi, "o", color=MLLMC, ms=9)
    ax.plot(SIGi.loc[e, "insp_axis"], yi, "s", color=BLUE, ms=8)
ax.axvline(0.5, ls=":", color="#999")
ax.text(0.5, len(order) - 0.3, "chance .50", ha="center", color="#999", fontsize=10)
ax.axhline(3.5, color="#1a1a1a", lw=0.8)
ax.set_yticks(y)
ax.set_yticklabels([LBL[e] for e in order])
ax.set_xlim(0.4, 0.95)
ax.set_xlabel("Axis accuracy")
ax.legend(
    handles=[
        Line2D([], [], marker="o", color=MLLMC, ls="", label="Sympathy axis", ms=9),
        Line2D([], [], marker="s", color=BLUE, ls="", label="Inspiration axis", ms=8)
    ],
    frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "fig_appendix_perdim.png"), dpi=250, bbox_inches="tight")
plt.close(fig)


## 11. Appendix Figure E5 — Valence separation across channels in the 100 Exp 4 explanations

In [12]:
mpl.rcdefaults(); plt.close("all")
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import os, sys, glob

def apply_figure_style(*, frame="open", font=None, sizes=(8, 7, 6), grid=False):
    if frame not in ("open", "boxed", "none"):
        raise ValueError(f"frame must be 'open'|'boxed'|'none', got {frame!r}")
    try:
        import os, sys, glob, matplotlib.font_manager as fm
        fdir = os.path.join(os.environ.get("CONDA_PREFIX") or sys.prefix, "fonts")
        if os.path.isdir(fdir):
            known = {f.fname for f in fm.fontManager.ttflist}
            for f in glob.glob(os.path.join(fdir, "*.ttf")):
                if f not in known:
                    fm.fontManager.addfont(f)
    except Exception:
        pass
    base, secondary, tick = sizes
    boxed = (frame == "boxed")
    rc = {
        "font.family": "sans-serif",
        "font.size": base,
        "axes.labelsize": base,
        "axes.titlesize": base,
        "legend.fontsize": secondary,
        "xtick.labelsize": tick,
        "ytick.labelsize": tick,
        "axes.linewidth": 0.6,
        "xtick.direction": "out", "ytick.direction": "out",
        "xtick.major.size": 3, "ytick.major.size": 3,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "axes.spines.top": boxed, "axes.spines.right": boxed,
        "axes.spines.left": frame != "none", "axes.spines.bottom": frame != "none",
        "axes.grid": bool(grid),
        "legend.frameon": False,
        "figure.dpi": 200,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    }
    mpl.rcParams.update(rc)

apply_figure_style()

BLUE, ORANGE, GREY, GOLD = "#3B6FB0", "#E08A2B", "#8C8C8C", "#C9A227"

VAL = pd.read_csv(os.path.join(REPRO, "exp4_valence_by_channel_predicted.csv"))

plt.close("all")
fig, ax = plt.subplots(figsize=(6.4, 4.6))
x = np.arange(3)
w = 0.32
CH = ["text", "nontext", "physio"]
XL = ["Text", "Nonverbal", "Physiological"]
V = VAL.set_index("channel").loc[CH]
ax.bar(x - w/2, V.mean_SYMP, w, color=ORANGE, label=f"Sympathy (n = {int(V.n_SYMP.iloc[0])})")
ax.bar(x + w/2, V.mean_INSP, w, color=GREY, label=f"Inspiration (n = {int(V.n_INSP.iloc[0])})")
ax.axhline(0, color="#333", lw=0.8)
for i, p in enumerate(V.p):
    ax.text(i, 1.5, "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s.", ha="center", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(XL)
ax.set_ylim(-1.3, 1.8)
ax.set_ylabel("Valence tendency\n(positive − negative cues per explanation)")
ax.set_title("Valence separation is carried by language and, more weakly, nonverbal cues;\nphysiological cues are frequent but non-discriminative", loc="left")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "figE5_valence_by_channel.png"), dpi=250)
plt.close(fig)
